In [0]:
import os, json, urllib.request

RAW = "/Volumes/workspace/pobreza/raw/banco_mundial"
os.makedirs(RAW, exist_ok=True)

PAISES = ["URY", "ARG", "BRA", "CHL", "PRY", "CRI", "MEX", "LCN", "OED"]
INDICADORES = ["NY.GDP.PCAP.PP.KD", "NY.GDP.MKTP.KD.ZG", "FP.CPI.TOTL.ZG", "SL.UEM.TOTL.ZS",
               "SL.UEM.1524.ZS", "SI.POV.GINI", "SI.POV.UMIC", "VC.IHR.PSRC.P5", "SE.SEC.CUAT.UP.ZS"]

for ind in INDICADORES:
    url = (f"https://api.worldbank.org/v2/country/{';'.join(PAISES)}/indicator/{ind}"
           f"?format=json&date=2000:2025&per_page=5000")
    with urllib.request.urlopen(url) as r:
        datos = json.loads(r.read())
    with open(f"{RAW}/{ind}.json", "w") as f:
        json.dump(datos[1], f)
    print(ind, len(datos[1]), "filas")


In [0]:
from pyspark.sql import functions as F

filas = []
for ind in INDICADORES:
    for r in json.load(open(f"{RAW}/{ind}.json")):
        filas.append((ind, r["indicator"]["value"], r["countryiso3code"], r["country"]["value"], r["date"], None if r["value"] is None else float(r["value"])))

bronze = spark.createDataFrame(filas, "indicador string, indicador_nombre string, iso3 string, pais_en string, anio string, valor double")
bronze.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.pobreza.banco_mundial_bronze")
print(bronze.count(), "filas")
     